# LogiCrisis — GRPO Training on Colab
**Meta PyTorch OpenEnv Hackathon | Theme #1: Multi-Agent Interactions**

Stack: `Unsloth` 4-bit QLoRA + `TRL GRPOTrainer` + `LogiCrisisEnv`

> **Runtime**: GPU required — Runtime → Change runtime type → T4 GPU (free) or A100

## Step 1 — Install dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "trl>=0.9.0" datasets accelerate peft
!pip install -q fastapi uvicorn pydantic gradio httpx

## Step 2 — Load the LogiCrisis codebase

Upload the `logicriasis/` folder to your Google Drive root, then run:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
LOGICRIASIS_PATH = '/content/drive/MyDrive/logicriasis'
sys.path.insert(0, LOGICRIASIS_PATH)
os.chdir(LOGICRIASIS_PATH)
print('Working directory:', os.getcwd())

from environment.tasks import ALL_TASK_IDS
print('Tasks loaded:', ALL_TASK_IDS)

## Step 3 — Heuristic baseline (BEFORE training)
Run this once. Save these numbers — this is your **before** score.

In [ ]:
from inference import run_episode
from environment.tasks import ALL_TASK_IDS

print('=== HEURISTIC BASELINE (no LLM) ===')
baseline_scores = {}
for task_id in ALL_TASK_IDS:
    result = run_episode(task_id, use_llm=False)
    baseline_scores[task_id] = result['score']
    status = '✓ PASS' if result['passed'] else '✗ FAIL'
    print(f'  {task_id:<35} score={result["score"]:.4f}  {status}')

avg = sum(baseline_scores.values()) / len(baseline_scores)
print(f'\n  Average heuristic: {avg:.4f}')
print('\n  ← Save these numbers. This is your BEFORE.')

## Step 4 — Load model with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/llama-3-8b-instruct-bnb-4bit"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print('Model loaded:', MODEL_NAME)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Step 5 — Build prompt dataset

In [ ]:
from training.train import build_prompt_dataset

dataset = build_prompt_dataset(n_samples=128, curriculum_level=1)
print(f'Dataset: {len(dataset)} prompts')
print('\nSample prompt (first 500 chars):')
print(dataset[0]['prompt'][:500])

## Step 6 — Reward function sanity check

In [ ]:
from training.train import _score_completion

good = '{"action_type": "reroute", "cargo_id": "C001", "route_id": "Mumbai-Pune", "reasoning": "Rerouting around flood zone via open road"}'
mid  = '{"action_type": "wait"}'
bad  = 'I will reroute the cargo to Mumbai'

print(f'Good action:      {_score_completion(good):+.3f}  (expect ~+0.6)')
print(f'Wait no reason:   {_score_completion(mid):+.3f}  (expect ~+0.2)')
print(f'Bad (no JSON):    {_score_completion(bad):+.3f}  (expect -0.5)')

## Step 7 — GRPO Training

**T4**: ~30–40 min | **A100**: ~10 min

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.train import grpo_reward_fn

grpo_config = GRPOConfig(
    output_dir="/content/logicriasis_outputs",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    logging_steps=5,
    save_steps=50,
    report_to="none",
    max_completion_length=256,
    num_generations=4,
    temperature=0.7,
    seed=42,
    fp16=True,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[grpo_reward_fn],
    args=grpo_config,
    train_dataset=dataset,
)

print('Starting GRPO training…')
trainer.train()
print('Training complete!')

## Step 8 — Save LoRA adapters

In [ ]:
SAVE_DIR = "/content/logicriasis_outputs/final"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Adapters saved to {SAVE_DIR}')

# Copy to Drive so they survive session restart
import shutil
drive_save = '/content/drive/MyDrive/logicriasis_adapters'
shutil.copytree(SAVE_DIR, drive_save, dirs_exist_ok=True)
print(f'Also saved to Drive: {drive_save}')

## Step 9 — Enable fast inference on the trained model

This switches the model from training mode to inference mode (2× faster generation).

In [ ]:
import torch, json
from unsloth import FastLanguageModel
from environment.models import AgentAction, ActionType
from agents.prompts import get_system_prompt, build_user_prompt

# Switch model to fast inference mode (required after training)
FastLanguageModel.for_inference(model)
print('Model switched to inference mode ✓')

def generate_action(agent_id: str, obs) -> AgentAction:
    """Call the trained model to pick an action for one agent."""
    messages = [
        {"role": "system", "content": get_system_prompt(obs.role.value)},
        {"role": "user",   "content": build_user_prompt(obs.to_prompt_text())},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    # Strip markdown fences if model wraps output
    if response.startswith("```"):
        response = "\n".join(
            l for l in response.split("\n") if not l.strip().startswith("```")
        )

    try:
        data = json.loads(response)
        atype = ActionType(data.get("action_type", "wait"))
        return AgentAction(
            agent_id=agent_id, action_type=atype,
            cargo_id=data.get("cargo_id"),
            route_id=data.get("route_id"),
            target_region=data.get("target_region"),
            bid_price=data.get("bid_price"),
            bid_capacity=data.get("bid_capacity"),
            target_agent=data.get("target_agent"),
            bid_id=data.get("bid_id"),
            coalition_id=data.get("coalition_id"),
            coalition_members=data.get("coalition_members"),
            coalition_role=data.get("coalition_role"),
            reward_split=data.get("reward_split"),
            reasoning=data.get("reasoning", ""),
        )
    except (json.JSONDecodeError, ValueError):
        return AgentAction(agent_id=agent_id, action_type=ActionType.WAIT,
                           reasoning="parse_error")


def run_task_with_llm(task_id: str) -> dict:
    """Run a full episode using the trained LLM and return the grade dict."""
    from environment.tasks import get_task
    task = get_task(task_id)
    env  = task.make_env(seed=42)
    observations = env.reset()

    for _ in range(env.world.max_turns):
        actions = {aid: generate_action(aid, obs)
                   for aid, obs in observations.items()}
        result = env.step(actions)
        observations = result.observations
        if result.terminated or result.truncated:
            break

    grade = task.grade(env)
    return grade

print('generate_action() and run_task_with_llm() ready ✓')

## Step 10 — Before / After comparison (THE MONEY SLIDE)

This runs all 9 tasks with the **trained LLM** and compares against the heuristic baseline from Step 3.

In [ ]:
from environment.tasks import ALL_TASK_IDS

print('=== GRPO-TRAINED LLM vs HEURISTIC BASELINE ===')
print(f'{"Task":<35} {"Heuristic":>10} {"LLM":>10} {"Delta":>8} {"Status"}')
print('-' * 75)

llm_scores = {}
for task_id in ALL_TASK_IDS:
    result = run_task_with_llm(task_id)
    llm_score = result['score']
    heur_score = baseline_scores.get(task_id, 0.0)
    llm_scores[task_id] = llm_score
    delta = llm_score - heur_score
    status = '✓ PASS' if result['passed'] else '✗ FAIL'
    sign = '+' if delta >= 0 else ''
    print(f'{task_id:<35} {heur_score:>10.4f} {llm_score:>10.4f} {sign+f"{delta:.4f}":>8}  {status}')

print('-' * 75)
avg_heur = sum(baseline_scores.values()) / len(baseline_scores)
avg_llm  = sum(llm_scores.values()) / len(llm_scores)
delta    = avg_llm - avg_heur
sign     = '+' if delta >= 0 else ''
print(f'{"AVERAGE":<35} {avg_heur:>10.4f} {avg_llm:>10.4f} {sign+f"{delta:.4f}":>8}')
print()
print(f'  Heuristic pass rate: {sum(1 for s in baseline_scores.values() if s >= 0.35)}/9')
print(f'  LLM pass rate:       {sum(1 for r in llm_scores.values() if r >= 0.35)}/9')
print(f'  Average improvement: {sign}{delta:.4f}')

## Step 11 — Sample LLM actions (show the panel)

Print the raw LLM reasoning for earthquake_relief and capacity_crunch — the two hard tasks.

In [ ]:
from environment.tasks import get_task

for demo_task_id in ['earthquake_relief', 'capacity_crunch']:
    print(f'\n{"="*60}')
    print(f'TASK: {demo_task_id}')
    print(f'{"="*60}')

    task = get_task(demo_task_id)
    env = task.make_env(seed=42)
    observations = env.reset()

    # Show first 3 turns of LLM actions
    for turn_num in range(3):
        print(f'\n--- Turn {turn_num + 1} ---')
        actions = {}
        for agent_id, obs in observations.items():
            action = generate_action(agent_id, obs)
            actions[agent_id] = action
            print(f'  [{agent_id}] {action.action_type.value}')
            if action.cargo_id:
                print(f'    cargo: {action.cargo_id}  route: {action.route_id}')
            if action.bid_price:
                print(f'    bid_price: {action.bid_price}  target: {action.target_agent}')
            if action.coalition_members:
                print(f'    coalition: {action.coalition_members}')
            if action.reasoning:
                print(f'    reasoning: "{action.reasoning[:100]}"')

        result = env.step(actions)
        observations = result.observations
        print(f'  → OTIF after turn: {result.info["otif_percent"]}%')
        if result.terminated or result.truncated:
            break

    grade = task.grade(env)
    print(f'\nFinal score: {grade["score"]:.4f}  {grade["verdict"]}')

## Step 12 — Optional: Launch Gradio demo in Colab

In [ ]:
import subprocess, threading

def run_demo():
    subprocess.run(['python', 'demo/app.py'])

t = threading.Thread(target=run_demo, daemon=True)
t.start()
print('Demo starting — check output above for Gradio share link (valid 72h)')